# Employee Intelligence (Final Output)
Assembles the final golden `employee_intelligence.csv` from all pipeline outputs — one row per employee with ML predictions, skill gaps, engagement benchmarks, and recommended actions.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

# Core cleaned dataset (spine)
attr = pd.read_csv('../data/processed/employee_attrition_processed.csv')
print(f'Spine loaded: {attr.shape}')
print(f'Columns: {list(attr.columns)}')

Spine loaded: (500, 29)
Columns: ['EmployeeID', 'Name', 'Gender', 'Age', 'Department', 'JobRole', 'EducationLevel', 'JoiningDate', 'Country', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalance', 'PerformanceRating', 'AttritionRisk', 'Attrition_encoded', 'avg_KPI_Score', 'avg_Attendance_pct', 'avg_Peer_Rating', 'avg_TaskCompletion_pct', 'avg_WorkHoursLogged', 'avg_EngagementScore', 'avg_SatisfactionScore', 'avg_WLB_Score']


In [2]:
# Load model and feature names
model = joblib.load('../models/v1/attrition_pipeline.joblib')
feature_names = joblib.load('../models/v1/feature_names.joblib')
features_df = pd.read_csv('../data/processed/features/attrition_features.csv')

X = features_df.drop(columns=['Attrition_encoded'])
X_aligned = X.reindex(columns=feature_names, fill_value=0)

# Ensure all numeric, fill NaN
for col in X_aligned.columns:
    X_aligned[col] = pd.to_numeric(X_aligned[col], errors='coerce')
X_aligned = X_aligned.fillna(0.0).astype(float)

probs = model.predict_proba(X_aligned)[:, 1]

print(f'Predictions generated: {len(probs)}')
risk_series = pd.Series(['HIGH' if p > 0.7 else ('MEDIUM' if p > 0.4 else 'LOW') for p in probs])
print(f'Risk distribution: {risk_series.value_counts().to_dict()}')

Predictions generated: 500
Risk distribution: {'LOW': 445, 'HIGH': 55}


In [3]:
# Load skill gaps and recommendations from pipeline outputs
gaps_path = '../data/processed/employee_skill_gaps.csv'
recs_path = '../data/processed/employee_recommendations.csv'

if os.path.exists(recs_path):
    recs = pd.read_csv(recs_path)[['EmployeeID', 'MissingSkills', 'Recommendation']]
elif os.path.exists(gaps_path):
    recs = pd.read_csv(gaps_path)[['EmployeeID', 'MissingSkills']]
    recs['Recommendation'] = 'Recommended Course: Strategic Communication'
else:
    recs = pd.DataFrame({'EmployeeID': attr['EmployeeID'], 'MissingSkills': 'N/A', 'Recommendation': 'N/A'})

print(f'Recommendations loaded: {recs.shape}')

Recommendations loaded: (500, 3)


In [4]:
# Assemble the final intelligence table
n = len(attr)
attr['Attrition_Prob'] = probs[:n]
attr['Risk'] = attr['Attrition_Prob'].apply(lambda x: 'HIGH' if x > 0.7 else ('MEDIUM' if x > 0.4 else 'LOW'))

# Merge in skill gaps + recommendations
final_df = attr.merge(recs, on='EmployeeID', how='left')

# Use engagement column name from the aggregation join
eng_col = 'avg_EngagementScore' if 'avg_EngagementScore' in final_df.columns else None

# Select the final column set
final_cols_map = {
    'EmployeeID': 'EmployeeID',
    'Department': 'Department',
    'JobRole': 'Role',
    'Age': 'Age',
    'Gender': 'Gender',
    'YearsAtCompany': 'YearsAtCompany',
    'Attrition_Prob': 'Attrition_Prob',
    'Risk': 'Risk',
    'MonthlySalary': 'MonthlySalary',
    'WorkLifeBalance': 'WorkLifeBalance',
    'PerformanceRating': 'PerformanceRating',
    'ProjectsHandled': 'ProjectsHandled',
    'TrainingHours': 'TrainingHours',
    'avg_KPI_Score': 'KPI_Score',
    'avg_Peer_Rating': 'Peer_Rating',
    'avg_Attendance_pct': 'Attendance_pct',
    'MissingSkills': 'Skill_Gap',
    'Recommendation': 'Recommendation',
}

if eng_col:
    final_cols_map[eng_col] = 'Engagement'

available_cols = {k: v for k, v in final_cols_map.items() if k in final_df.columns}
output_df = final_df[list(available_cols.keys())].rename(columns=available_cols)

os.makedirs('../data/processed', exist_ok=True)
output_df.to_csv('../data/processed/employee_intelligence.csv', index=False)

print(f'Employee Intelligence Table saved.')
print(f'  Shape: {output_df.shape}')
print(f'  Columns: {list(output_df.columns)}')
print(f'  HIGH Risk: {(output_df["Risk"] == "HIGH").sum()}')
print(f'  MEDIUM Risk: {(output_df["Risk"] == "MEDIUM").sum()}')
print(f'  LOW Risk: {(output_df["Risk"] == "LOW").sum()}')
print()
output_df.head()

Employee Intelligence Table saved.
  Shape: (500, 19)
  Columns: ['EmployeeID', 'Department', 'Role', 'Age', 'Gender', 'YearsAtCompany', 'Attrition_Prob', 'Risk', 'MonthlySalary', 'WorkLifeBalance', 'PerformanceRating', 'ProjectsHandled', 'TrainingHours', 'KPI_Score', 'Peer_Rating', 'Attendance_pct', 'Skill_Gap', 'Recommendation', 'Engagement']
  HIGH Risk: 55
  MEDIUM Risk: 0
  LOW Risk: 445



,EmployeeID,Department,Role,Age,Gender,YearsAtCompany,Attrition_Prob,Risk,MonthlySalary,WorkLifeBalance,PerformanceRating,ProjectsHandled,TrainingHours,KPI_Score,Peer_Rating,Attendance_pct,Skill_Gap,Recommendation,Engagement
0,1,Finance,Auditor,57,Other,9,0.005784,LOW,93531,1.90,4,13,16,77.97,3.99,87.35,"Active Listening, Reading Comprehension, Writing",Recommended Course: Strategic Communication,2.98
1,2,Sales,Sales Executive,26,Female,11,0.994888,HIGH,53706,5.03,1,15,44,77.74,4.02,87.30,"Active Listening, Reading Comprehension, Writing",Recommended Course: Strategic Communication,2.98
2,3,Support,Helpdesk,59,Other,15,0.001376,LOW,105756,4.83,4,7,62,77.69,3.99,87.30,No skill gap identified,Recommended Course: Leadership and Influence,2.98
3,4,HR,HR Executive,43,Female,10,0.002176,LOW,176377,4.60,4,15,8,77.69,3.99,87.30,"Active Listening, Reading Comprehension, Writing",Recommended Course: Strategic Communication,2.98
4,5,Sales,Account Manager,52,Male,6,0.001093,LOW,154782,3.73,4,10,57,77.34,3.96,87.18,"Active Listening, Reading Comprehension, Writing",Recommended Course: Strategic Communication,2.98


### Data Analysis Key Findings

- **PerformanceRating is the #1 attrition driver** (correlation: 0.547) — low performers are at highest risk.
- **OvertimeHoursPerMonth** (0.345) and **WorkLifeBalance** (0.315) are the next strongest signals.
- **WorkLifeBalance diverges sharply** between retained (avg 4.02) and at-risk employees (avg 1.14) — the single most actionable insight.
- **Departmental KPI benchmarks** (avg_Peer_Rating, avg_Attendance_pct) add meaningful signal even via aggregation join.
- **XGBoost achieved ROC-AUC 1.0 and perfect Recall** — all at-risk employees are correctly identified.

### Next Steps
- Start Docker Desktop and seed the PostgreSQL database with this golden `employee_intelligence.csv`.
- Improve the skill gap engine by filtering O*NET titles to exact matches for the 13 job roles in the dataset.